# 🚀 Two-Stage Linear Evaluation: Hierarchical SupCon + Linear Probe (NO CRF)

Notebook này thực hiện huấn luyện và đánh giá mô hình **PhoBERT-base** trên tác vụ **PAP_NER** theo chuẩn **Two-Stage Linear Evaluation Protocol** (Google Khosla et al., 2020) kết hợp **Hierarchical Supervised Contrastive Learning (H-SupCon, $\alpha = 0.3$)** và **Linear Probe (hoàn toàn KHÔNG dùng CRF)**.

### 📌 Cấu hình thực nghiệm:
- **Giai đoạn 1 (Stage 1):** Pre-training PhoBERT với H-SupCon ($\alpha = 0.3, \tau = 0.07$), batch size = 64, LR = 2e-5, Projection Head LR = 1e-3, Early stopping = 5 epochs.
- **Giai đoạn 2 (Stage 2):** Đóng băng 100% Backbone PhoBERT, chỉ huấn luyện tầng `nn.Linear(768, 11)` với Cross-Entropy Loss (KHÔNG CÓ CRF). Suy luận: `argmax(logits)`.
- **Đánh giá:** CoNLL-2003 Strict Entity-level Micro-F1 (đếm chuẩn 1x) + Token-level Classification Report.

## Bước 1: Kiểm tra GPU và môi trường

In [ ]:
!nvidia-smi

## Bước 2: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q transformers>=4.28.0 scikit-learn pandas matplotlib tqdm py_vncorenlp

## Bước 3: Thực thi toàn trình Stage 1 + Stage 2 (Two-Stage Pipeline)

In [ ]:
!python main.py \
    --task pap_ner \
    --data_dir ./Dataset \
    --model_name_or_path vinai/phobert-base \
    --output_dir outputs/phobert_base_hsupcon_linear_eval \
    --stage1_epochs 15 \
    --stage1_batch_size 64 \
    --stage1_lr 2e-5 \
    --stage1_proj_lr 1e-3 \
    --supcon_temp 0.07 \
    --supcon_alpha 0.3 \
    --proj_dim 128 \
    --stage1_early_stop 5 \
    --stage2_epochs 20 \
    --stage2_batch_size 64 \
    --stage2_lr 1e-3 \
    --stage2_early_stop 5 \
    --seed 42 \
    --fp16

## Bước 4: Hiển thị Ma trận nhầm lẫn (Normalized Confusion Matrix)

In [ ]:
from IPython.display import Image, display
import os

cm_path = 'outputs/phobert_base_hsupcon_linear_eval/confusion_matrix.png'
if os.path.exists(cm_path):
    display(Image(filename=cm_path))
else:
    print('Chưa tìm thấy file ma trận nhầm lẫn. Vui lòng kiểm tra lại output_dir!')